# 03 — Stability of the loading subspace

Thesis Chapter 5. Is the FFM loading subspace stable over 2014–2025? Loadings are only identified up to rotation,
so stability is studied geometrically, through the **Grassmann distance** between loading subspaces estimated
on rolling 6-month windows, compared with a simulated null distribution, and then tested formally
(CUSUM, Bai-Perron, Dette & Kutta).

Requires the shared objects saved by `01_data_eda_ffm_rolling.ipynb` (`data/shared/`).

## Contents

1. [Imports & shared objects](#Imports-&-shared-objects)
2. [Geometry](#Geometry) — per-pair distance, distance to the 2014 window, pairwise heatmap, CUSUM, Bai-Perron
3. [Simulated null distribution of $d_G$](#Simulated-null-distribution-of-$d_G$)
4. [Mean loading across rolling windows](#Mean-loading-across-rolling-windows)
5. [Dette & Kutta](#Dette-&-Kutta)

# Import

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
from functools import partial

In [ ]:
# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import acf

In [ ]:
# ── Data processing & factor model ───────────────────────────────────────────

from ffm_fx_real import run_ffm
from factor_model_fcts import build_panel_from_matrices

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────

In [ ]:
# ── Forecasting ───────────────────────────────────────────────────────────────

In [ ]:
# ── Stability ─────────────────────────────────────────────────────────────────
from ffm_stability import (
    grassman_rolling_window_ref,
    plot_grassman,
    plot_loadings_mean_evolution,
    grassman_rolling_by_pair,
    plot_distance_distribution,
    plot_cusum,
    plot_bai_perron_distances,
    compute_cusum_ma5,
    bai_perron_distances_hac,
)

In [ ]:
from dg_simulation import run_simulation_dg, plot_simulation_dg

In [ ]:
%load_ext autoreload
%autoreload 2

# Imports & shared objects

Load the objects computed and saved by `01_data_eda_ffm_rolling.ipynb`.

In [ ]:
# ── Load shared objects (saved by 01_data_eda_ffm_rolling) ────────────────────
import joblib

shared_dir = "../data/shared/"

ffm              = joblib.load(f"{shared_dir}ffm.pkl")
vol_ffm_all      = joblib.load(f"{shared_dir}vol_ffm_all.pkl")
vol_matrices_all = joblib.load(f"{shared_dir}vol_matrices_all.pkl")
avg_vol          = joblib.load(f"{shared_dir}avg_vol.pkl")
FX_all           = joblib.load(f"{shared_dir}FX_all.pkl")

print(f"ffm loaded  — T={ffm['U_hat'].shape[1]} days, N={len(ffm['pairs'])} pairs")
print(f"vol_ffm_all — {len(vol_ffm_all)} pairs")

# Geometry

### Grassmann distance per pair (vs 2014)

In [ ]:
grass_by_pair = grassman_rolling_by_pair(
    data        = vol_ffm_all,
    window_size = 6,
    step_months        = 1,
    func = lambda x, mu_ref=None: run_ffm(x, k_max=5, r_forced=1, silent=True, mu_ref=mu_ref),
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
    ref_start   = '2014-01-01',
)

## Grassmann distance to the 2014 reference window

In [ ]:
period = [6]
step = [30]

for p in period:
    for s in step:
        print(f"Computing window={p}M, step={s}d")
        if p == 3: end = '2025-10-01'
        if p == 6: end = '2025-07-01'
        if p == 12: end = '2025-01-01'
        results = grassman_rolling_window_ref(
            vol_ffm_all, 
            window_size = p, 
            step = s, 
            start_date = '2014-01-01', 
            end_date = end 
        )
        results.to_parquet(f"../data/results/grass_dist_{p}_{s}_mu.parquet", engine='pyarrow')



In [ ]:
from ffm_stability import grassmann_distance_matrix, plot_grassmann_heatmap

In [ ]:

%load_ext autoreload
%autoreload 2

In [ ]:
dist_matrix, labels = grassmann_distance_matrix(
    vol_ffm_all,
    window_size=6,
    step=6,
    start_date='2014-01-01',
    end_date='2025-12-31'
)

plot_grassmann_heatmap(
    dist_matrix, labels,
    save_path='../figures/STABILITY/grassmann_heatmap_pairwise.png'
)

In [ ]:
print(dg_ref_6M_30d.head())

### 6-month windows, 30-day step

In [ ]:
dg_ref_6M_30d = pd.read_parquet("../data/results/grass_dist_6_30_mu.parquet", engine='pyarrow')

In [ ]:
print(dg_ref_6M_30d.nlargest(1, 'distance_grass'))

In [ ]:
print(dg_ref_6M_30d.nlargest(5, 'distance_grass'))

In [ ]:
plot_grassman(dg_ref_6M_30d, avg_vol, 6, 30, reference=True) 

In [ ]:
# single plot
plot_grassman(dg_ref_6M_30d, avg_vol, 6, 30, reference=True, show_dynamics=False)


In [ ]:
plot_distance_distribution(dg_ref_6M_30d, label='6M - 30d')

### CUSUM (HAC long-run variance)

In [ ]:
result_6M_30d, cp_date, cv = compute_cusum_ma5(dg_ref_6M_30d, confidence=0.95)
plot_cusum(result_6M_30d, cp_date, cv, window_label='6M / 30d')

### Bai-Perron (BIC, HAC)

In [ ]:
bp_6M_30d = bai_perron_distances_hac(
    dg_ref_6M_30d,
    col        = 'distance_grass',
    date_col   = 'start',
    max_breaks = 10,
    h_frac     = 0.10,   # minimum regime length ≈ 12 months (13 of 134 windows)
    criterion  = 'bic',
)
plot_bai_perron_distances(bp_6M_30d)

In [ ]:
print(dg_ref_6M_30d['distance_grass'].mean())

# Simulated null distribution of $d_G$

Two independent samples of size S = 132 from the reference covariance $\Sigma_{ref}$ (Jan–Jun 2014), B = 1000 replications.

In [ ]:
dg_simulation = run_simulation_dg(vol_ffm_all, start_date='2014-01-02', end_date='2014-07-01', S=132, B=1000, r=1)

In [ ]:
pd.DataFrame({'distance_grass': dg_simulation}).to_parquet(f"../data/results/dg_simulation.parquet", engine='pyarrow')

In [ ]:
dg_simulation = pd.read_parquet("../data/results/dg_simulation.parquet", engine='pyarrow')
plot_simulation_dg(dg_simulation)

# Mean loading across rolling windows

In [ ]:
run_ffm_r1 = partial(run_ffm, r_forced=1, silent= True)

mean_loadings_arr, dates = plot_loadings_mean_evolution(
    data        = vol_ffm_all,
    window_size = 6,
    step        = 30,
    func        = run_ffm_r1,
    start_date  = '2014-01-01',
    end_date    = '2025-07-01',
    r_plot      = 2,
)

# Dette & Kutta

Relevant-change tests on the eigensystem of the stacked FFM panel.

Reference: Dette & Kutta (2021), *Electronic Journal of Statistics*.

The panel is stacked: Y ∈ ℝ^{T × NM} where N = number of pairs, M = number of intraday intervals.
All functions assume centred data (cf. eq. 2.1 of the paper).

## 1. Panel construction

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd

from ffm_fx_real  import build_panel_from_matrices
from dette_kutta  import (
    stack_panel,
    estimate_change_point,
    estimate_change_point_eigenvalue,
    estimate_change_point_eigenfunction,
    eigenvalue_test,
    eigenfunction_test,
)

X_3d, days, tau, pairs = build_panel_from_matrices(vol_ffm_all, min_coverage=0.8)
Y, delta = stack_panel(X_3d)

print(f"Panel: T={Y.shape[0]} days  D={Y.shape[1]} (={len(pairs)} pairs × {X_3d.shape[2]} intervals)")


## 2. Global estimator — full covariance operator

CUSUM on the Frobenius norm of the full operator (Eq. 2.6).
Detects the dominant break across all directions.

In [ ]:
cp_global = estimate_change_point(Y, eps=0.05, dates=days)
print(f"Break global : {str(cp_global['break_date'])[:10]}  (θ̂ = {cp_global['theta_hat']:.3f},  k₀ = {cp_global['k0']})")


## 3. Specific estimators — eigenvalue and eigenfunction

They maximise, respectively:
- `f_ev(k) = [k(N-k)/N²] · (τ₁^(1)(k) − τ₁^(2)(k))²`
- `f_ef(k) = [k(N-k)/N²] · ‖v₁^(1)(k) − v₁^(2)(k)‖²_L²`

Grid `step=20` (~1 month) to limit computation time.

In [ ]:
cp_ev = estimate_change_point_eigenvalue(Y, j=1, eps=0.05, step=20, dates=days)
cp_ef = estimate_change_point_eigenfunction(Y, j=1, eps=0.05, step=20, dates=days)

print(f"Break eigenvalue   : {str(cp_ev['break_date'])[:10]}  (θ̂ = {cp_ev['theta_hat']:.3f},  k₀ = {cp_ev['k0']})")
print(f"Break eigenfunction: {str(cp_ef['break_date'])[:10]}  (θ̂ = {cp_ef['theta_hat']:.3f},  k₀ = {cp_ef['k0']})")


## 4. Eigenfunction test — break 2018-11-16

We test H₀ : ‖v₁^(1) − v₁^(2)‖²_L² ≤ Δv with Δv = 0 (the most demanding choice) and the break fixed at the estimated date.

In [ ]:
res_ef_trade = eigenfunction_test(
    Y,
    j          = 1,
    Delta_v    = 0.0,
    alpha      = 0.05,
    K          = 20,
    break_date = cp_ef["break_date"],
    plot       = True,
    pair_name  = "panel_ef_trade",
    dates      = days,
    delta=delta
)

print(f"Break : {str(cp_ef['break_date'])[:10]}")
print(f"D̂  = {res_ef_trade['D_hat']:.5f}   Û = {res_ef_trade['U_hat']:.5f}")
print(f"stat = {res_ef_trade['stat']:.3f}   q = {res_ef_trade['q_crit']:.3f}   p-value = {res_ef_trade['p_value']:.4f}")
print(f"Reject H₀ : {res_ef_trade['reject']}")


## 5. Tests on the joint break — 2022-02-17

The CUSUM of the full covariance operator identifies Feb. 2022 as the dominant break.
We test whether it is carried by the leading eigenvalue τ₁, the leading eigenfunction v₁, or both (Δ = 0).

In [ ]:
res_ev_ukraine = eigenvalue_test(
    Y, j=1, Delta_tau=0.0, alpha=0.05, K=20,
    break_date = cp_global["break_date"],
    plot=True, pair_name="panel_ev_ukraine", dates=days,delta=delta
)

res_ef_ukraine = eigenfunction_test(
    Y, j=1, Delta_v=0.0, alpha=0.05, K=20,
    break_date = cp_global["break_date"],
    plot=True, pair_name="panel_ef_ukraine", dates=days,delta=delta
)

print(f"Break : {str(cp_global['break_date'])[:10]}")
print(f"[Eigenvalue]    τ^(1)={res_ev_ukraine['tau1_j']:.5f}  τ^(2)={res_ev_ukraine['tau2_j']:.5f}  E^={res_ev_ukraine['E_hat']:.5f}  V^={res_ev_ukraine['V_hat']:.5f}. stat={res_ev_ukraine['stat']:.3f}  p={res_ev_ukraine['p_value']:.4f}  Reject={res_ev_ukraine['reject']}")
print(f"[Eigenfunction] D̂={res_ef_ukraine['D_hat']:.5f}  Û={res_ef_ukraine['U_hat']:.5f}  stat={res_ef_ukraine['stat']:.3f}  p={res_ef_ukraine['p_value']:.4f}  Reject={res_ef_ukraine['reject']}")
